In [ ]:
import os, subprocess, sys, time
os.chdir('/kaggle/working')
print('ETAPE1: clone + deps')
r = subprocess.run(['git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git'],capture_output=True,text=True)
print('clone rc=', r.returncode, r.stderr[-300:] if r.returncode else '')
r = subprocess.run([sys.executable,'-m','pip','install','-q','-r','ComfyUI/requirements.txt'],capture_output=True,text=True)
print('req rc=', r.returncode, r.stderr[-300:] if r.returncode else '')
r = subprocess.run([sys.executable,'-m','pip','install','-q','aiohttp'],capture_output=True,text=True)
print('aiohttp rc=', r.returncode)
print('ETAPE1 FINIE')

In [ ]:
print('ETAPE2: checkpoint SDXL')
import os, subprocess
d='/kaggle/working/ComfyUI/models/checkpoints'
os.makedirs(d, exist_ok=True)
dst=os.path.join(d,'sd_xl_base_1.0.safetensors')
if not os.path.exists(dst):
    r = subprocess.run(['wget','-q','--show-progress','-O',dst,
      'https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors'],capture_output=True,text=True)
    print('wget rc=', r.returncode, (r.stderr or '')[-200:])
print('ckpt size:', os.path.getsize(dst) if os.path.exists(dst) else 'ABSENT')
print('ETAPE2 FINIE')


In [ ]:
# ETAPE2: modeles Hunyuan3D-2 (DiT fp16 + VAE fp16, ~8Go total)
import os, subprocess
dm = '/kaggle/working/ComfyUI/models/diffusion_models'
va = '/kaggle/working/ComfyUI/models/vae'
os.makedirs(dm, exist_ok=True); os.makedirs(va, exist_ok=True)
files = [
  (dm, 'hunyuan3d-dit-v2-0-fp16.safetensors',
   'https://huggingface.co/tencent/Hunyuan3D-2/resolve/main/hunyuan3d-dit-v2-0/model.fp16.safetensors'),
  (dm, 'hunyuan3d-dit-v2-0-turbo-fp16.safetensors',
   'https://huggingface.co/tencent/Hunyuan3D-2/resolve/main/hunyuan3d-dit-v2-0-turbo/model.fp16.safetensors'),
  (va, 'hunyuan3d-vae-v2-0-fp16.safetensors',
   'https://huggingface.co/tencent/Hunyuan3D-2/resolve/main/hunyuan3d-vae-v2-0/model.fp16.safetensors'),
]
for folder, name, url in files:
    dst = os.path.join(folder, name)
    if not os.path.exists(dst):
        r = subprocess.run(['wget','-q','--show-progress','-O',dst,url],capture_output=True,text=True)
        print('wget', name, 'rc=', r.returncode)
    print(name, 'size:', os.path.getsize(dst) if os.path.exists(dst) else 'ABSENT')
print('ETAPE2 FINIE')


In [ ]:
print('ETAPE4: tunnel')
import subprocess, re, time, threading, os
if not os.path.exists('/kaggle/working/cloudflared'):
    r = subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O','/kaggle/working/cloudflared'],capture_output=True,text=True)
    print('cloudflared dl rc=', r.returncode)
    subprocess.run(['chmod','+x','/kaggle/working/cloudflared'],check=False)
    _cf_cmd = (['/kaggle/working/cloudflared','tunnel','--no-autoupdate','run','--token',''] if '' else ['/kaggle/working/cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:8188'])
proc = subprocess.Popen(_cf_cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = [None]
def pump():
    for line in proc.stdout:
        print('CF:', line.strip()[:160])
        m = re.search('https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m and not url[0]: url[0] = m.group(0)
threading.Thread(target=pump, daemon=True).start()
for _ in range(40):
    if url[0]: break
    time.sleep(2)
print('>>> KAGGLE_COMFY_URL =', url[0])

In [ ]:
# WATCHDOG + KEEP-ALIVE : relance cloudflared ET comfy s'ils meurent
# -> surveille ce log : a chaque (re)demarrage du tunnel la NOUVELLE URL s'affiche
import time, subprocess, os, signal, re, threading
cf = [None]
def start_cf():
    _cf_cmd = (['/kaggle/working/cloudflared','tunnel','--no-autoupdate','run','--token',''] if '' else ['/kaggle/working/cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:8188'])
    cf[0] = subprocess.Popen(_cf_cmd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    def pump():
        for line in cf[0].stdout:
            m = re.search('https://[a-z0-9-]+\.trycloudflare\.com', line)
            if m:
                print('>>> TUNNEL ACTIF:', m.group(0), flush=True)
            elif ' ERR ' in line or 'error' in line.lower():
                print('CF-ERR:', line.strip()[:140], flush=True)
    threading.Thread(target=pump, daemon=True).start()
def start_comfy():
    global comfy
    comfy = subprocess.Popen([sys.executable,'main.py','--listen','127.0.0.1','--port','8188','--disable-cuda-malloc'],
        cwd='/kaggle/working/ComfyUI')
print('watchdog actif - /shutdown ou kaggle kernels stop pour arreter', flush=True)
while True:
    if cf[0] is None or cf[0].poll() is not None:
        print('WATCHDOG: cloudflared mort -> redemarrage', flush=True)
        start_cf()
    if comfy.poll() is not None:
        print('WATCHDOG: comfy mort -> redemarrage', flush=True)
        start_comfy()
    time.sleep(20)


In [ ]:
print('ETAPE4: tunnel')
import subprocess, re, time, threading, os
if not os.path.exists('/kaggle/working/cloudflared'):
    r = subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O','/kaggle/working/cloudflared'],capture_output=True,text=True)
    print('cloudflared dl rc=', r.returncode)
    subprocess.run(['chmod','+x','/kaggle/working/cloudflared'],check=False)
    _cf_cmd = (['/kaggle/working/cloudflared','tunnel','--no-autoupdate','run','--token',''] if '' else ['/kaggle/working/cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:8188'])
proc = subprocess.Popen(_cf_cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = [None]
def pump():
    for line in proc.stdout:
        print('CF:', line.strip()[:160])
        m = re.search('https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m and not url[0]: url[0] = m.group(0)
threading.Thread(target=pump, daemon=True).start()
for _ in range(40):
    if url[0]: break
    time.sleep(2)
print('>>> KAGGLE_COMFY_URL =', url[0])